# День 3 — Baseline-модель классификации

**Цель:** построить первую воспроизводимую модель классификации финансовых текстов и зафиксировать её качество как точку отсчёта.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from finnews_sentiment.data.load_data import DEFAULT_CONFIG, load_financial_phrasebank
from finnews_sentiment.features.preprocess import EXPECTED_SENTIMENTS, prepare_news_data
from finnews_sentiment.models.train_model import save_baseline_artifacts, train_baseline

## 1. Конфигурация

Ноутбук автономен: он не читает CSV, созданные в дни 1–2. Исходный датасет загружается общим модулем проекта, а preprocessing выполняется напрямую в памяти.

In [2]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'pyproject.toml').exists() else current_dir.parent
assert (PROJECT_ROOT / 'pyproject.toml').exists(), 'Запустите ноутбук из корня проекта или папки notebooks'

MODEL_PATH = PROJECT_ROOT / 'models/baseline_logreg.joblib'
METRICS_PATH = PROJECT_ROOT / 'reports/day03/baseline_metrics.json'
TEST_SIZE = 0.2
RANDOM_STATE = 42

## 2. Загрузка и подготовка данных

Используем конфигурацию `sentences_75agree`. Очистка удаляет пустые строки и дубли, нормализует текст и проверяет метки классов.

In [3]:
raw_df = load_financial_phrasebank(DEFAULT_CONFIG)
df = prepare_news_data(raw_df)

print(f'Исходных строк: {len(raw_df)}')
print(f'После preprocessing: {len(df)}')
display(df.head())

Исходных строк: 3453
После preprocessing: 3448


,text,sentiment,text_clean,label,word_count_clean,char_count,dollar_count
0,"According to Gran , the company has no plans t...",neutral,"according to gran , the company has no plans t...",1,25,127,0
1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2,33,206,0
2,"For the last quarter of 2010 , Componenta 's n...",positive,"for the last quarter of 2010 , componenta 's n...",2,39,193,0
3,"In the third quarter of 2010 , net sales incre...",positive,"in the third quarter of 2010 , net sales incre...",2,29,125,0
4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2,24,122,0


In [4]:
class_distribution = pd.DataFrame({
    'Количество': df['sentiment'].value_counts().reindex(EXPECTED_SENTIMENTS),
    'Доля, %': (df['sentiment'].value_counts(normalize=True).reindex(EXPECTED_SENTIMENTS) * 100).round(2),
})
display(class_distribution)

,Количество,"Доля, %"
sentiment,,
negative,420,12.18
neutral,2141,62.09
positive,887,25.73


Нейтральный класс заметно преобладает, поэтому используем стратифицированное разбиение и оцениваем baseline прежде всего по **macro F1** — эта метрика даёт одинаковый вес каждому классу.

## 3. Обучение baseline

Pipeline объединяет TF-IDF с униграммами и биграммами и Logistic Regression. Векторизатор обучается только на train-части, поэтому информация из test не попадает в признаки.

In [5]:
model, metrics = train_baseline(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    'train': metrics['split']['train_distribution'],
    'test': metrics['split']['test_distribution'],
}).reindex(EXPECTED_SENTIMENTS)
display(split_summary)

,train,test
negative,336,84
neutral,1713,428
positive,709,178


## 4. Метрики на test set

In [8]:
report = pd.DataFrame(metrics['classification_report']).T
display(report.round(4))

summary = pd.DataFrame(
    {'Значение': [metrics['accuracy'], metrics['macro_f1'], metrics['weighted_f1']]},
    index=['Accuracy', 'Macro F1', 'Weighted F1'],
).round(4)
display(summary)

,precision,recall,f1-score,support
negative,0.9545,0.5000,0.6562,84.0000
neutral,0.8210,0.9860,0.8960,428.0000
positive,0.8409,0.6236,0.7161,178.0000
accuracy,0.8333,0.8333,0.8333,0.8333
macro avg,0.8722,0.7032,0.7561,690.0000
weighted avg,0.8424,0.8333,0.8204,690.0000


,Значение
Accuracy,0.8333
Macro F1,0.7561
Weighted F1,0.8204


In [9]:
assert len(df) == 3448
assert metrics['split']['train_rows'] == 2758
assert metrics['split']['test_rows'] == 690
assert set(model.classes_) == set(EXPECTED_SENTIMENTS)
assert 0 <= metrics['macro_f1'] <= 1
print('Проверки baseline пройдены.')

Проверки baseline пройдены.


## 5. Сохранение baseline

Сохраняем весь pipeline вместе с TF-IDF, а метрики — также в читаемом JSON. Эти результаты станут точкой отсчёта для улучшений дня 4.

In [10]:
save_baseline_artifacts(model, metrics, MODEL_PATH, METRICS_PATH)
print(f'Модель: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'Метрики: {METRICS_PATH.relative_to(PROJECT_ROOT)}')

Модель: models/baseline_logreg.joblib
Метрики: reports/day03/baseline_metrics.json


## 6. Выводы

In [14]:
display(Markdown(
    f"- Baseline **TF-IDF + Logistic Regression** достиг macro F1 **{metrics['macro_f1']:.4f}**.\n"
    f"- Accuracy составила **{metrics['accuracy']:.4f}**.\n"
    "- Результат сохранён и будет использоваться для сравнения следующих моделей.\n"
    "- Из-за дисбаланса классов улучшения следует оценивать по macro F1, а не только по accuracy."
))

- Baseline **TF-IDF + Logistic Regression** достиг macro F1 **0.7561**.
- Accuracy составила **0.8333**.
- Результат сохранён и будет использоваться для сравнения следующих моделей.
- Из-за дисбаланса классов улучшения следует оценивать по macro F1, а не только по accuracy.